# ER1: No-Communication Control

Multi-agent coordination emerges from two sources: **what agents learn** and **what agents share**.
Before measuring the value of communication, we need to isolate the first factor — how well can agents learn to coordinate through observation alone?

This notebook trains agents on the Discovery K-N convergence task with **zero communication channels**.
The result is a performance floor: any communication protocol (ER2–E1) that fails to beat this floor adds complexity without benefit.

> **Research question:** Under partial observability and no message passing, how much of the rendezvous task can MARL policies solve through individual learning alone?

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
RENDEZVOUS_ROOT = REPO_ROOT / "rendezvous_comm"
sys.path.insert(0, str(RENDEZVOUS_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_experiment, find_configs
from src.storage import ExperimentStorage
from src.runner import run_sweep, evaluate_with_vmas, make_heuristic_policy_fn
from src.display import (
    display_config, display_metrics, display_sweep_summary, display_run_status,
    display_environment_info, scrollable, display_metric_cards, display_verdict,
    display_config_selector,
)
from src.plotting import (
    plot_sweep_heatmap, plot_seed_variance, plot_training_dashboard,
    plot_baseline_comparison_bars, save_figure,
)

print(f"Torch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

# Show available configs with completion status and freshness
display_config_selector("er1")

## Configuration

Select a config from the table above. Each YAML is self-contained — filename encodes what's inside:
- `single_*` → 1 run for quick validation (~15 min on CPU)
- `sweep_*` → full parameter sweep (hours/days)

Freshness badges: **VALID** = results match current config+code, **CONFIG/CODE CHANGED** = results are stale, **NEW** = no results yet.

In [ ]:
# ── Select config here ──
CONFIG = RENDEZVOUS_ROOT / "configs" / "er1" / "single_mappo_n4_l035.yaml"
FORCE_RETRAIN = False  # Set True to re-run even if results exist

spec = load_experiment(CONFIG)
spec.ensure_dirs()

display_environment_info(spec)
display_config(spec)

## Measurement Strategy

We evaluate along three axes relevant to the no-comm baseline:

| Category | Metric | What it tells us |
|----------|--------|-----------------|
| **Task performance** | M1 Success Rate | Did agents cover *all* targets? |
| | M2 Avg Return | Net reward (covering − collisions − time) |
| | M3 Avg Steps | Speed of completion (200 = never finished) |
| | M6 Coverage Progress | Partial credit: fraction of targets covered |
| **Safety** | M4 Collisions/Episode | Spatial awareness between agents |
| **Coordination** | M8 Agent Utilization | Workload balance (CV, 0 = perfectly balanced) |
| | M9 Spatial Spread | Mean pairwise distance (higher = exploring) |

**M5 (Tokens)** is always 0 here — it becomes meaningful in ER2+.
**M7 (Sample Efficiency)** is computed post-training from BenchMARL curves.

### Reference baselines
Before training, we measure two non-learned policies to calibrate expectations:

In [ ]:
SANITY_OVERRIDES = {
    "n_agents": 4, "n_targets": 7, "agents_per_target": 2,
    "targets_respawn": False,
}

heuristic_fn = make_heuristic_policy_fn()
heuristic_metrics = evaluate_with_vmas(
    spec.task, task_overrides=SANITY_OVERRIDES,
    policy_fn=heuristic_fn, n_eval_episodes=200, n_envs=200,
)

random_metrics = evaluate_with_vmas(
    spec.task, task_overrides=SANITY_OVERRIDES,
    policy_fn=None, n_eval_episodes=200, n_envs=200,
)

# Side-by-side comparison
fig = plot_baseline_comparison_bars(
    {"Random": random_metrics, "Heuristic": heuristic_metrics},
    metric_key="M1_success_rate",
    title="Pre-Training Reference: Success Rate",
)
plt.show()

fig = plot_baseline_comparison_bars(
    {"Random": random_metrics, "Heuristic": heuristic_metrics},
    metric_key="M6_coverage_progress",
    title="Pre-Training Reference: Coverage Progress",
)
plt.show()

display_metrics(heuristic_metrics, title="Heuristic Baseline (4 agents, 7 targets, K=2)")
display_metrics(random_metrics, title="Random Baseline (4 agents, 7 targets, K=2)")

## Training

Each run: freeze config → train with BenchMARL (MAPPO/IPPO) → save policy → evaluate → save metrics + report.
Set `skip_complete=True` to resume interrupted sweeps without re-running finished runs.

In [ ]:
display_run_status(spec)
results = run_sweep(spec, skip_complete=not FORCE_RETRAIN)

## Training Curves

BenchMARL logs scalar metrics at each iteration. The dashboard below shows four key signals:
- **Eval Reward** — are agents improving?
- **Targets Covered** — are they learning the actual objective?
- **Covering Reward** — reward component from covering targets
- **Policy Entropy** — exploration level (should decrease but not collapse)

In [ ]:
storage = ExperimentStorage("er1")
all_metrics = storage.load_all_metrics()

if not all_metrics:
    print("No completed runs yet. Run the training cell first.")
else:
    # Training curves from BenchMARL CSVs
    latest_run_id = sorted(all_metrics.keys())[-1]
    run_storage = storage.get_run(latest_run_id)
    scalars = run_storage.load_benchmarl_scalars()

    if scalars:
        fig = plot_training_dashboard(
            scalars,
            title=f"Training Progress — {latest_run_id}",
            heuristic_reward=heuristic_metrics.get("M2_avg_return"),
        )
        save_figure(fig, str(run_storage.output_dir / "training_dashboard.png"))
        plt.show()
    else:
        print("No BenchMARL scalars found (training may not have completed).")

## Results

In [ ]:
if not all_metrics:
    print("No completed runs yet.")
else:
    latest_run_id = sorted(all_metrics.keys())[-1]
    trained_metrics = all_metrics[latest_run_id]
    run_storage = storage.get_run(latest_run_id)

    # ── KPI cards ──
    display_metric_cards(trained_metrics, title=f"Trained Policy — {latest_run_id}")

    # ── Three-way comparison ──
    comparison = {
        "Random": random_metrics,
        "Heuristic": heuristic_metrics,
        "Trained (MAPPO)": trained_metrics,
    }
    comparison_colors = {
        "Random": "#e74c3c", "Heuristic": "#27ae60", "Trained (MAPPO)": "#1f77b4"
    }

    for metric_key in ["M1_success_rate", "M2_avg_return", "M6_coverage_progress"]:
        fig = plot_baseline_comparison_bars(
            comparison, metric_key=metric_key, colors=comparison_colors,
        )
        save_figure(fig, str(run_storage.output_dir / f"comparison_{metric_key}.png"))
        plt.show()

    # ── Full report (scrollable) ──
    report_path = run_storage.run_dir / "report.md"
    if report_path.exists():
        scrollable(report_path.read_text(), height=400, title="Full Run Report")

## Interpretation

The no-comm baseline tells us:
- **If M1 is low** (< 30%): the task is hard enough that communication should matter. Good experimental design.
- **If M1 is high** (> 60%): agents solve the task without communication — the task may be too easy, or the config too forgiving.
- **M6 vs M1 gap**: when M1 ≈ 0 but M6 > 0.5, agents learn to cover *most* targets but fail on the last few. Communication could help here.
- **M8 (utilization)**: high CV means some agents do all the work. Communication should improve coordination.
- **M9 (spread)**: low spread means agents clump. Communication can encourage exploration.

In [ ]:
if not all_metrics:
    print("No data available.")
else:
    df = storage.to_dataframe()
    if "n_agents" in df.columns:
        n4_df = df[df["n_agents"] == 4]
    else:
        n4_df = df

    if not n4_df.empty:
        avg_success = n4_df["M1_success_rate"].mean()
        avg_return = n4_df["M2_avg_return"].mean()
        display_verdict(avg_success, avg_return)

## Sweep Analysis

With the `complete.yaml` config, we can compare success across agent counts, LiDAR ranges, and algorithms.
If you used `fast.yaml`, this section will be skipped (only 1 run).

In [ ]:
if len(all_metrics) > 1 and not df.empty:
    # Success rate heatmap across configs
    if "n_agents" in df.columns and len(df["n_agents"].unique()) > 1:
        fig = plot_sweep_heatmap(
            df, metric="M1_success_rate",
            row_param="n_agents", col_param="lidar_range",
            title="ER1: Success Rate (N agents × LiDAR range)",
        )
        save_figure(fig, str(spec.results_dir / "sweep_success_heatmap.png"))
        plt.show()

    # Algorithm comparison
    if "algorithm" in df.columns and len(df["algorithm"].unique()) > 1:
        fig = plot_seed_variance(
            df, metric="M1_success_rate", group_by="algorithm",
            title="ER1: MAPPO vs IPPO Success Rate (across seeds)",
        )
        save_figure(fig, str(spec.results_dir / "sweep_algo_comparison.png"))
        plt.show()

    # Full summary table
    display_sweep_summary(all_metrics)
else:
    print("Sweep analysis requires multiple runs. Use configs/er1/complete.yaml and re-run.")

In [ ]:
# Show where results are stored
if all_metrics:
    latest_run_id = sorted(all_metrics.keys())[-1]
    run_storage = storage.get_run(latest_run_id)
    print(f"Run directory:   {run_storage.run_dir}")
    print(f"Policy file:     {run_storage.output_dir / 'policy.pt'}")
    print(f"Metrics file:    {run_storage.output_dir / 'metrics.json'}")
    print(f"Report:          {run_storage.run_dir / 'report.md'}")
    print()

    if run_storage.has_policy():
        size_kb = (run_storage.output_dir / "policy.pt").stat().st_size / 1024
        print(f"Policy saved: {size_kb:.1f} KB")
    print()

    # Example: reload and verify
    print("── Reload example ──")
    state_dict = run_storage.load_policy_state_dict()
    if state_dict:
        print(f"Loaded state_dict with {len(state_dict)} parameter tensors")
        total_params = sum(p.numel() for p in state_dict.values())
        print(f"Total parameters: {total_params:,}")
    print()

    # List all output artifacts
    print("── Output artifacts ──")
    for f in sorted(run_storage.output_dir.rglob("*")):
        if f.is_file():
            size = f.stat().st_size
            sz = f"{size/1024:.1f} KB" if size > 1024 else f"{size} B"
            rel = f.relative_to(run_storage.run_dir)
            print(f"  {rel}  ({sz})")
else:
    print("No completed runs.")

## Results Storage & Policy Export

Each run is self-contained in its timestamped folder:
```
results/er1/YYYYMMDD_HHMM__<run_id>/
  input/
    config.yaml           ← frozen config (reproducible)
    provenance.json       ← config+code hashes for freshness checks
  logs/run.log            ← full training log
  output/
    metrics.json          ← M1–M9 evaluation metrics
    policy.pt             ← trained policy weights
    training_dashboard.png
    comparison_*.png
    benchmarl/            ← raw BenchMARL artifacts
  report.md               ← markdown report with embedded images
```

Config files use descriptive names matching their parameters:
```
configs/er1/single_mappo_n4_l035.yaml           ← 1 run, quick validation
configs/er1/sweep_mappo-ippo_n2-6_l025-045.yaml ← full 120-run sweep
```

### Exporting a trained policy
Copy the run folder or just `policy.pt` + `config.yaml`.

### Importing / reloading a trained policy